# MLP complexity crossover: Baseline vs Adaptive + IPOPT

目标：固定类别数和 simplex 网格，只增加 MLP 的深度/宽度，观察单次梯度成本增大后，Adaptive Bundle + IPOPT 的梯度节省是否能抵消其外层优化开销。

近似时间模型：$T_B=N_BC_g+O_B$，$T_A=N_AC_g+O_I$。当 $(N_B-N_A)C_g>O_I-O_B$ 时，Adaptive 的墙钟时间可能优于 Baseline。每个 case 继续输出 `GN* vs time` 和 `GN* vs gradient evaluations` 两个 panel。

## 运行规则

1. Torch 与 cyipopt 在当前 macOS 环境有 OpenMP 加载顺序要求。请重启 kernel 后从本 notebook 第一格开始运行，不要提前 `import algorithm` 或 `import cyipopt`。
2. 配置单元只需修改一行 `RUN_MODE`：`quick` 验证流程，`fast` 依次运行三个小模型，`full` 执行完整 sweep。
3. 所有结果写入 `Adaptive Bundle Algorithm/output/complex_mlp_crossover/`。
4. 图中的 CPU 轴目前实际是 elapsed wall-clock time；本实验固定使用 CPU，不使用 GPU。

In [ ]:
from pathlib import Path
import csv
import json
import sys

import matplotlib.pyplot as plt
import numpy as np

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd / 'Adaptive Bundle Algorithm' / 'Original_py',
    cwd / '.venv' / 'First-order-method-smooth-MOO' / 'Adaptive Bundle Algorithm' / 'Original_py',
]
ORIGINAL_PY = next((p for p in candidates if (p / 'experiments.py').exists()), None)
if ORIGINAL_PY is None:
    raise FileNotFoundError('Could not locate Adaptive Bundle Algorithm/Original_py')
if str(ORIGINAL_PY) not in sys.path:
    sys.path.insert(0, str(ORIGINAL_PY))

# experiments imports objectives_torch before algorithm/cyipopt.
from experiments import experiment_mlp_gn_coverage, mlp_parameter_count

OUTPUT_BASE = ORIGINAL_PY.parent / 'output' / 'complex_mlp_crossover'

print('Original_py:', ORIGINAL_PY)
print('Output base:', OUTPUT_BASE)

In [ ]:
# 只修改这一行：'quick'、'fast' 或 'full'。
RUN_MODE = 'fast'

if RUN_MODE == 'quick':
    EXPECTED_RUNTIME = '约 10 秒'
    K, p, n = 2, 8, 40
    ARCHITECTURES = [[8], [12, 12]]
    SEEDS = [10]
    COARSE_RESOLUTION = 1
    N_PASSES = 1
    STEPS_PER_POINT = 1
    BASELINE_MAX_GRAD_EVALS = 4
    ADAPTIVE_MAX_GRAD_EVALS = 4
    EVAL_EVERY_N_GRADS = 2
    MAX_OUTER, MAX_INNER = 1, 1
    LAMBDA_MAX_STARTS = 2
    L_N_PROBES = 1
    ORACLE_BENCHMARK_REPEATS = 1
elif RUN_MODE == 'fast':
    EXPECTED_RUNTIME = '每个约 15–45 分钟；三个串行约 65–95 分钟（空闲 CPU 上）'
    # 三个受控宽度档位；除 hidden_sizes 外，其余实验条件完全相同。
    # 保持参数维度较小，避免 GN/IPOPT 被巨大梯度矩阵拖垮；
    # 用较大的样本数提高每次神经网络前向/反向传播的成本。
    K, p, n = 2, 20, 50_000
    ARCHITECTURES = [[64, 64], [80, 80], [96, 96]]
    SEEDS = [10]
    COARSE_RESOLUTION = 10
    N_PASSES = 1000  # max_grad_evals is the effective hard stop
    STEPS_PER_POINT = 5
    # Baseline 定义共同 GN* 目标；Adaptive 有更大的安全预算并在达标时停止。
    BASELINE_MAX_GRAD_EVALS = 100
    ADAPTIVE_MAX_GRAD_EVALS = 800
    EVAL_EVERY_N_GRADS = 100
    MAX_OUTER, MAX_INNER = 80, 5
    # K=2 时使用中心点和两个角点，避免单起点一直困在中心附近。
    LAMBDA_MAX_STARTS = 3
    L_N_PROBES = 1
    ORACLE_BENCHMARK_REPEATS = 1
elif RUN_MODE == 'full':
    EXPECTED_RUNTIME = '数小时到数天；不要用于试跑'
    # Hold K fixed so the first sweep isolates neural-network gradient cost.
    K, p, n = 5, 100, 1000
    ARCHITECTURES = [
        [32],
        [128, 128],
        [256, 256, 256],
        [512, 512, 512, 512],
    ]
    SEEDS = [10, 20, 30]
    COARSE_RESOLUTION = 5
    N_PASSES = 1000  # max_grad_evals is the effective hard stop
    STEPS_PER_POINT = 5
    BASELINE_MAX_GRAD_EVALS = 30_000
    ADAPTIVE_MAX_GRAD_EVALS = 30_000
    EVAL_EVERY_N_GRADS = 2_000
    MAX_OUTER, MAX_INNER = 2_000, 25
    LAMBDA_MAX_STARTS = 64
    L_N_PROBES = 5  # exploratory; increase for the final reported run
    ORACLE_BENCHMARK_REPEATS = 10
else:
    raise ValueError("RUN_MODE must be 'quick', 'fast', or 'full'")

OUTPUT_ROOT = OUTPUT_BASE / RUN_MODE
CURVE_DIR = OUTPUT_ROOT / 'curves'
HISTORY_DIR = OUTPUT_ROOT / 'histories'
for directory in (OUTPUT_ROOT, CURVE_DIR, HISTORY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print('Mode:', RUN_MODE.upper())
print('Expected wall time:', EXPECTED_RUNTIME)
print('Run output:', OUTPUT_ROOT)
for arch in ARCHITECTURES:
    print(f'  hidden_sizes={arch}, d={mlp_parameter_count(K, p, arch):,}')

In [ ]:
def _json_ready(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'Not JSON serialisable: {type(value)!r}')

def _history_payload(result):
    payload = {}
    for key in ('baseline', 'algorithm2'):
        run = result[key]
        payload[key] = {
            name: run[name]
            for name in ('cov_history', 'grad_evals_history', 'cpu_times')
        }
    return payload

records = []
results = {}

for hidden_sizes in ARCHITECTURES:
    arch_tag = 'x'.join(map(str, hidden_sizes))
    for seed in SEEDS:
        case = f'K{K}_p{p}_n{n}_h{arch_tag}_seed{seed}'
        print('\n' + '=' * 80)
        print('Running', case)

        result = experiment_mlp_gn_coverage(
            verbose=True,
            K=K, p=p, n=n, hidden_sizes=hidden_sizes, seed=seed,
            init_seed=seed + 1,
            coarse_resolution=COARSE_RESOLUTION,
            n_passes=N_PASSES,
            steps_per_point_per_pass=STEPS_PER_POINT,
            eval_every_n_grads=EVAL_EVERY_N_GRADS,
            max_grad_evals=None,
            baseline_max_grad_evals=BASELINE_MAX_GRAD_EVALS,
            adaptive_max_grad_evals=ADAPTIVE_MAX_GRAD_EVALS,
            max_outer=MAX_OUTER, max_inner=MAX_INNER,
            lambda_max_starts=LAMBDA_MAX_STARTS,
            lambda_solver='ipopt', require_ipopt=True,
            prune_inner=False,
            l_n_probes=L_N_PROBES,
            oracle_benchmark_repeats=ORACLE_BENCHMARK_REPEATS,
            run_baseline=True, run_adaptive=True,
            out_path=str(CURVE_DIR / f'{case}.png'),
        )
        results[case] = result
        baseline = result['baseline']
        adaptive = result['algorithm2']
        target = float(baseline['cov_history'][-1])
        adaptive_final = float(adaptive['cov_history'][-1])
        reached = adaptive_final <= target
        baseline_time = float(baseline['cpu_times'][-1])
        adaptive_time = float(adaptive['cpu_times'][-1])
        baseline_grads = int(baseline['grad_evals_history'][-1])
        adaptive_grads = int(adaptive['grad_evals_history'][-1])

        record = {
            'case': case, 'seed': seed, 'K': K, 'p': p, 'n': n,
            'hidden_sizes': list(hidden_sizes), 'd': int(result['d']),
            'baseline_max_grad_evals': BASELINE_MAX_GRAD_EVALS,
            'adaptive_max_grad_evals': ADAPTIVE_MAX_GRAD_EVALS,
            'lambda_max_starts': LAMBDA_MAX_STARTS,
            'oracle_ms': float(result['oracle_ms']),
            'problem_setup_time': float(result['problem_setup_time']),
            'target_gn': target, 'adaptive_final_gn': adaptive_final,
            'adaptive_reached_target': bool(reached),
            'baseline_time': baseline_time, 'adaptive_time': adaptive_time,
            'baseline_grad_evals': baseline_grads,
            'adaptive_grad_evals': adaptive_grads,
            'time_ratio_baseline_over_adaptive': (
                baseline_time / adaptive_time if reached and adaptive_time > 0 else None
            ),
            'grad_ratio_baseline_over_adaptive': (
                baseline_grads / adaptive_grads if reached and adaptive_grads > 0 else None
            ),
            'curve_plot': result['plot'],
        }
        records.append(record)
        if reached:
            print(f'VALID comparison: both methods reached GN* <= {target:.4e}')
        else:
            print(
                f'INVALID comparison: Adaptive GN*={adaptive_final:.4e} '
                f'did not reach the Baseline target {target:.4e}; '
                'CPU ratio will not be reported.'
            )

        with (HISTORY_DIR / f'{case}.json').open('w', encoding='utf-8') as f:
            json.dump(_history_payload(result), f, indent=2, default=_json_ready)
        with (OUTPUT_ROOT / 'summary.json').open('w', encoding='utf-8') as f:
            json.dump(records, f, indent=2, default=_json_ready)

print(f'\nCompleted {len(records)} runs.')

In [ ]:
csv_path = OUTPUT_ROOT / 'summary.csv'
csv_fields = [
    'case', 'seed', 'K', 'p', 'n', 'hidden_sizes', 'd',
    'baseline_max_grad_evals', 'adaptive_max_grad_evals',
    'lambda_max_starts', 'oracle_ms',
    'problem_setup_time', 'target_gn', 'adaptive_final_gn',
    'adaptive_reached_target', 'baseline_time', 'adaptive_time',
    'baseline_grad_evals', 'adaptive_grad_evals',
    'time_ratio_baseline_over_adaptive',
    'grad_ratio_baseline_over_adaptive', 'curve_plot',
]
with csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields)
    writer.writeheader()
    for row in records:
        csv_row = dict(row)
        csv_row['hidden_sizes'] = 'x'.join(map(str, row['hidden_sizes']))
        writer.writerow(csv_row)

header = (
    f"{'architecture':>16} {'d':>10} {'oracle ms':>12} "
    f"{'BL time':>12} {'A time':>12} {'BL/A time':>12} {'reached':>9}"
)
print(header)
for row in records:
    ratio = row['time_ratio_baseline_over_adaptive']
    ratio_text = '-' if ratio is None else f'{ratio:.3f}'
    arch = 'x'.join(map(str, row['hidden_sizes']))
    print(
        f"{arch:>16} {row['d']:>10,d} {row['oracle_ms']:>12.3f} "
        f"{row['baseline_time']:>12.3f} {row['adaptive_time']:>12.3f} "
        f"{ratio_text:>12} {str(row['adaptive_reached_target']):>9}"
    )
print('CSV:', csv_path)

In [ ]:
aggregate = []
for hidden_sizes in ARCHITECTURES:
    rows = [r for r in records if r['hidden_sizes'] == list(hidden_sizes)]
    valid_time = [r['time_ratio_baseline_over_adaptive'] for r in rows
                  if r['time_ratio_baseline_over_adaptive'] is not None]
    valid_grad = [r['grad_ratio_baseline_over_adaptive'] for r in rows
                  if r['grad_ratio_baseline_over_adaptive'] is not None]
    aggregate.append({
        'hidden_sizes': list(hidden_sizes),
        'd': rows[0]['d'],
        'oracle_ms': float(np.median([r['oracle_ms'] for r in rows])),
        'time_ratio': float(np.median(valid_time)) if valid_time else np.nan,
        'grad_ratio': float(np.median(valid_grad)) if valid_grad else np.nan,
        'successes': len(valid_time),
        'runs': len(rows),
    })

x = np.array([r['d'] for r in aggregate], dtype=float)
time_ratio = np.array([r['time_ratio'] for r in aggregate], dtype=float)
grad_ratio = np.array([r['grad_ratio'] for r in aggregate], dtype=float)
labels = ['x'.join(map(str, r['hidden_sizes'])) for r in aggregate]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
finite_time = np.isfinite(time_ratio)
finite_grad = np.isfinite(grad_ratio)
if finite_time.any():
    axes[0].plot(x[finite_time], time_ratio[finite_time], marker='o', color='#9467bd')
else:
    axes[0].text(0.5, 0.5, 'No case reached the common GN* target',
                 transform=axes[0].transAxes, ha='center', va='center')
axes[0].axhline(1.0, color='black', ls='--', lw=1)
axes[0].set_xscale('log')
axes[0].set_xlabel('MLP parameter count d')
axes[0].set_ylabel('Baseline / Adaptive wall time')
axes[0].set_title('CPU-time crossover (>1: Adaptive wins)')
axes[0].grid(alpha=0.25, which='both')

if finite_grad.any():
    axes[1].plot(x[finite_grad], grad_ratio[finite_grad], marker='o', color='#1f77b4')
else:
    axes[1].text(0.5, 0.5, 'No case reached the common GN* target',
                 transform=axes[1].transAxes, ha='center', va='center')
axes[1].axhline(1.0, color='black', ls='--', lw=1)
axes[1].set_xscale('log')
axes[1].set_xlabel('MLP parameter count d')
axes[1].set_ylabel('Baseline / Adaptive gradient evaluations')
axes[1].set_title('Gradient-efficiency ratio (>1: Adaptive wins)')
axes[1].grid(alpha=0.25, which='both')

positive_x = x[x > 0]
for ax, values in zip(axes, (time_ratio, grad_ratio)):
    if positive_x.size:
        ax.set_xlim(positive_x.min() * 0.8, positive_x.max() * 1.25)
    for xi, yi, label in zip(x, values, labels):
        if np.isfinite(yi):
            ax.annotate(label, (xi, yi), xytext=(4, 4), textcoords='offset points', fontsize=8)

summary_plot = OUTPUT_ROOT / 'complexity_crossover_summary.png'
fig.savefig(summary_plot, dpi=170, bbox_inches='tight')
plt.close(fig)

winners = [r for r in aggregate if np.isfinite(r['time_ratio']) and r['time_ratio'] > 1.0]
if winners:
    first = min(winners, key=lambda r: r['d'])
    print('First observed crossover:', first)
else:
    print('No CPU-time crossover observed in the completed cases.')
print('Summary plot:', summary_plot)

## 解释结果

每个 case 曲线图的右侧 panel 是 `GN* vs gradient evaluations`。绿色虚线是 Baseline 最终 GN*；如果蓝色 Adaptive 曲线到达或低于绿色虚线，就表示 `adaptive_reached_target=True`。

- `Baseline / Adaptive wall time > 1`：Adaptive + IPOPT 更快。
- `Baseline / Adaptive gradient evaluations > 1`：Adaptive 使用更少梯度。
- 只有 `adaptive_reached_target=True` 的 case 才进入时间/梯度比值汇总。
- 如果梯度比始终大于 1、但时间比小于 1，说明 IPOPT/bundle 开销尚未被梯度成本覆盖。
- 本实现使用 ReLU，因此 $L_i$ 是经验估计；`L_N_PROBES=5` 适合探索，不应直接当作严格理论上界。

In [ ]:
print('Generated files:')
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file():
        print(' -', path.relative_to(OUTPUT_ROOT))